In [ ]:
#Here we repeat the Appendix C experiment with negative sampling
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML

from BadData_AppC import DataObject, AppendixCFunction, Trainer, Overlap
from NegSamplingMath_Unlearn_AppC_FullDataSet import NegMLP, NegTrainer, ActPoly, accuracy, NegOverlap

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


class Yeeplot_modular:
    """Class for plotting modular function experiment results.

    data: single data points dictionary,list or tuple of data points to plot
    label: label or list of labels for the data
    x_label: label for x-axis
    y_label: label for y-axis
    latex_title: title in LaTeX format
    p: modulus
    title: title of the plot
    fontsize: font size for the plot"""

    def __init__(self, data, label, y_label, title, x_label="Training step", fontsize=10):
        self.data = data
        self.label = label
        self.x_label = x_label
        self.y_label = y_label
        self.fontsize = fontsize
        self.title = title

    def plot(self):
        plt.figure()
        try:
            if isinstance(self.data, list) and isinstance(self.label, list):
                try:
                    for d, l in zip(self.data, self.label):
                        plt.plot(d, label=l)
                except ValueError as ve:
                    print(f"ValueError during plotting: {ve}")
            else:
                plt.plot(self.data, label=self.label)
        except Exception as e:
            print(f"Error plotting data: {e}")
        plt.xlabel(self.x_label)
        plt.ylabel(self.y_label)
        plt.title(self.title)
        plt.legend()
        plt.show()

def run_and_visualize_experiment(config: dict):
    """
    Runs a single experiment based on a configuration dictionary and visualizes the results.
    """
    print("="*80)
    print(f"Starting Experiment: {config['latex_title']} (p={config['p']})")
    print("="*80)

    # 1. Setup
    modular_function = AppendixCFunction(config['c'], config['d'], config['p'])
    dataset = DataObject(modular_function, split=config['split'])
    model = NegMLP(config['p'], config['embedding_dim'], config['hidden'])
    model = model.to(DEVICE)
    trainer = NegTrainer(learning_rate=config['learning_rate'], num_negs_per_example=config['negs_per_ex'])
    
    num_params = sum(p.numel() for p in model.parameters())
    space_dim = (config['p'] ** 2) * 2**4
    pos_dim = config['p'] ** 2
    print(f"num_parameters: {num_params:,}; space_dim: {space_dim:,}; pos_dim: {pos_dim:,}")
    
    if config.get('print_test_len', False):
        print(len(dataset.test_data))

    # 2. Training
    trainer.train_model(
        model,
        dataset,
        max_steps=config['max_steps'],
        batch_size=config['batch_size'],
        weight_decay=config['weight_decay']
    )

    # Polynomial as a string
    latex_title = config['latex_title']
    p = config['p']

    # 3. Visualization 

    # Losses
    Yeeplot_modular(
        data=[model.loss_dictionary['train_loss'], model.loss_dictionary['test_loss']],
        label=["Train Loss", "Test Loss"],
        y_label="Loss",
        title=f"Loss {latex_title} mod {p}" #I recommend to use f-strings for better readability, this is very powerful tool
    ).plot()

    # Accuracies
    Yeeplot_modular(
        data=[model.loss_dictionary['train_accuracy'], model.loss_dictionary['test_accuracy']],
        label=["Train Accuracy", "Test Accuracy"],
        y_label="Accuracy",
        title=f"Accuracy {latex_title} mod {p}"
    ).plot()

    # Positive-example accuracies
    Yeeplot_modular(
        data=[model.loss_dictionary['positive_train_accuracy'], model.loss_dictionary['positive_test_accuracy']],
        label=["Train Accuracy on positive examples", "Test Accuracy on positive examples"],
        y_label="Accuracy",
        title=f"Accuracy on positive examples {latex_title} mod {p}"
    ).plot()

    # Negative-example accuracies
    Yeeplot_modular(
        data=[model.loss_dictionary['negative_train_accuracy'], model.loss_dictionary['negative_test_accuracy']],
        label=["Train Accuracy on negative examples", "Test Accuracy on negative examples"],
        y_label="Accuracy",
        title=f"Accuracy on negative examples {latex_title} mod {p}"
    ).plot()

    # Animation for single-point test sets
    if len(dataset.test_data) == 1:

        rcParams['animation.embed_limit'] = 64  # MB, default is 20

        # Histogram
        example = dataset.test_data
        print(example)
        plt.figure()
        plt.bar(list(range(p)), model.loss_dictionary['counts_hist_all'])
        plt.title("Histogram " + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
        plt.xlabel("Question index")
        plt.ylabel("Count of predictions")
        plt.draw()
        plt.pause(0.001)
        plt.show

        #Histogram with single prediction
        plt.figure()
        plt.bar(list(range(p)), model.loss_dictionary['prediction_mike'])
        plt.title("Histogram for single prediction " + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
        plt.xlabel("Question index")
        plt.ylabel("Count of predictions")
        plt.draw()
        plt.pause(0.001)
        plt.show

        # Evolution of probability distribution
        probs = []
        for f in model.loss_dictionary.get('sorta_prob_dist', []):
            arr = f.detach().cpu().squeeze().numpy() if isinstance(f, torch.Tensor) else np.array(f).squeeze()
            s = float(arr.sum())
            if s > 0:  
                arr = arr / s
            else:
                print("Warning: Sum of probabilities is zero.")
            probs.append(arr)

        if probs:
            fig, ax = plt.subplots(figsize=(6,3))
            ax.set_title('Probability distribution evolution ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)   
            bars = ax.bar(range(len(probs[0])), probs[0])
            ax.set_ylim(0, 0.025) 
            txt = ax.text(0.02, 0.95, '', transform=ax.transAxes)

            def update(i):
                y = probs[i]
                for b, h in zip(bars, y): 
                    b.set_height(float(h))
                txt.set_text(f"step {i+1}/{len(probs)} | sum={y.sum():.3f} | argmax={int(np.argmax(y))}")
                return bars
            
            anim = FuncAnimation(fig, update, frames=len(probs), interval=100, repeat=False)
            plt.close(fig)
            display(HTML(anim.to_jshtml()))
    
    print("\n✅ Experiment Complete.\n")


: 

In [ ]:
#Polynomials modulo 53

experiments = [
    {
    # Polynomial (4*x + y**2)**3 % 53
        "p": 53, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    },
    # Polynomial (4*x + y**2)**3 + xy % 53
    {
        "p": 53, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    },
    # Polynomial (2*x + 3*y)**4 % 53
    {
        "p": 53, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 53
    {
        "p": 53, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    },
    # Polynomial (*x**3 + 2*y**4)**2 % 53
    {
        "p": 53, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    },
    # Polynomial (*x**3 + 2*y**4)**2 - y % 53
    {
        "p": 53, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.99999, "negs_per_ex": 20
    }
    # test
]

default_training_params = {
    "max_steps": 1000,
    "learning_rate": 0.005,
    "batch_size": 1024,
    "weight_decay": 1e-4,
}


In [ ]:
for exp_config in experiments:
    final_config = {**default_training_params, **exp_config} # Merge dictionaries
    run_and_visualize_experiment(final_config)

In [ ]:
# Polynomial (4*x + y**2)**3 % 97

experiments = [
    {
        "p": 97, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.5,
    },
    # Polynomial (4*x + y**2)**3 + xy % 97
    {
        "p": 97, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (2*x + 3*y)**4 % 97
    {
        "p": 97, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 97
    {
        "p": 97, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 % 97
    {
        "p": 97, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 - y % 97
    {
        "p": 97, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 512, "hidden": 5000, "split": 0.9999999999,
    },
    # Polynomial (4*x + y**2)**3 % 23
    {
        "p": 23, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (4*x + y**2)**3 + xy % 23
    {
        "p": 23, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (2*x + 3*y)**4 % 23
    {
        "p": 23, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 23
    {
        "p": 23, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # Polynomial (5*x**3 + 2*y**4)**2 % 23
    {
        "p": 23, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    #Polynomial (5*x**3 + 2*y**4)**2 - y % 23
    {
        "p": 23, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 64, "hidden": 5000, "split": 0.999,
    },
    # test
]

default_training_params = {
    "max_steps": 1000,
    "learning_rate": 0.005,
    "batch_size": 1024,
    "weight_decay": 1e-3,
}


In [ ]:

for exp_config in experiments:
    final_config = {**default_training_params, **exp_config} # Merge dictionaries
    run_and_visualize_experiment(final_config)